# Completed processing output audit

**Historical investigation, not current terrain coverage.** The original Earth tile/halo defect was corrected in [notebook 09](09_verify_earth_coverage_fix.ipynb). Lunar source heights were finite; our image-support mask introduced the elevation holes. [Notebook 13](13_build_lunar_terrain_views.ipynb) supplies a full-context view with zero elevation gaps inside the seven ROIs. Read [notebook 14](../exploration/14_terrain_feature_eda.ipynb) for the corrected terrain and separate support views. Remaining derivative edge exclusions and genuine environmental missingness are different issues.

Read-only validation of the five pinned runs from notebooks 02?06. Writes a new timestamped folder under `reports/`. Does not rerun processing or modify raw/processed inputs. Run from the project root or a folder under notebooks with the project environment.

Checks full output hashes, grid and mask invariants, complete feature neighborhoods, native seam jumps, climate cell tables, OSM geometry, and all Diviner output tables. Obsolete standalone raster previews have been removed; the numerical audit remains.

These checks establish internal consistency, not independent source accuracy or physical suitability. Missing native units, geographic support differences, patch-scale validity, and spatially separated model evaluation require explicit downstream decisions. No automatic pass/fail scientific approval is implied.

**Workflow stage: preprocessing / data readiness.** Use this notebook to inspect coverage, investigate exclusions, or verify corrections before EDA. Its saved results and pinned inputs preserve the original investigation. Notebook 13 builds the two lunar terrain views; notebook 14 explores the corrected Earth run and those views. See the [workflow and acceptance checks](../README.md). Moving this notebook does not change when the original findings were made.


In [ ]:
import hashlib
import json
import math
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from shapely.geometry import box, shape

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'outline.md').exists())
RUNS = {
    'earth_terrain_landcover': '20261028T014314564317Z_8a3e5d',
    'earth_climate': '20261028T014917410405Z_d7c8d5',
    'osm_support': '20261028T015656094003Z_ec31c5',
    'lunar_terrain_environment': '20261028T022855014566Z_cffea7',
    'diviner_temperature': '20261028T022942126792Z_d44299',
}
OUT = ROOT / 'reports' / ('processing-audit-' + datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ'))
OUT.mkdir(parents=True, exist_ok=False)
records = {}
results = {'runs': RUNS, 'scope': 'Completed outputs only; no raw-data transformations or notebook02-06 execution.'}


def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))


def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def load_raster(path):
    with rasterio.open(path) as src:
        return src.read(1, masked=True).astype('float32').filled(np.nan)


def stats(values):
    finite = values[np.isfinite(values)]
    return {'cells': int(values.size), 'valid_fraction': float(finite.size / values.size),
            'min': float(finite.min()) if finite.size else None,
            'p01': float(np.quantile(finite, .01)) if finite.size else None,
            'median': float(np.median(finite)) if finite.size else None,
            'p99': float(np.quantile(finite, .99)) if finite.size else None,
            'max': float(finite.max()) if finite.size else None}


In [ ]:
# Integrity: compare every output against the hash recorded at run completion.
integrity = []
for stage, run_id in RUNS.items():
    directory = ROOT / 'data/processed' / stage / run_id
    record = read_json(directory / 'provenance.json')
    assert record['status'] == 'complete'
    records[stage] = record
    failures = []
    for item in record['outputs']:
        path = directory / item['path']
        if not path.is_file() or path.stat().st_size != item['bytes'] or digest(path) != item['sha256']:
            failures.append(item['path'])
    integrity.append({'stage': stage, 'outputs': len(record['outputs']), 'failures': failures})
    print('Integrity', stage, len(record['outputs']), 'failures', len(failures), flush=True)
results['integrity'] = integrity

In [ ]:
# Full-resolution grid, range, mask, and feature support checks.
raster_rows, terrain_rows, previews, native_rows = [], [], [], []
features = ['elevation_m', 'relative_elevation_m', 'slope_deg', 'laplacian_per_m', 'local_elevation_std_m']
for stage in ['earth_terrain_landcover', 'lunar_terrain_environment']:
    base = ROOT / 'data/processed' / stage / RUNS[stage]
    for grid_dir in sorted(base.glob('*/*/grid_30m')):
        region = grid_dir.parent.name
        arrays = {p.stem: load_raster(p) for p in grid_dir.glob('*.tif')}
        with rasterio.open(grid_dir / 'elevation_m.tif') as src:
            reference = (src.shape, src.transform, src.crs)
            cell_area = abs(src.transform.a * src.transform.e)
        mismatched = []
        for p in sorted(grid_dir.glob('*.tif')):
            with rasterio.open(p) as src:
                if (src.shape, src.transform, src.crs) != reference:
                    mismatched.append(p.name)
            row = {'stage': stage, 'region': region, 'layer': p.stem, **stats(arrays[p.stem])}
            raster_rows.append(row)
        z = arrays['elevation_m']
        joint = np.logical_and.reduce([np.isfinite(arrays[f]) for f in features])
        row = {'stage': stage, 'region': region, 'grid_mismatches': mismatched,
               'joint_feature_fraction_of_grid': float(joint.mean()),
               'joint_feature_area_km2': float(joint.sum() * cell_area / 1e6),
               'slope_outside_0_90': int(((arrays['slope_deg'] < 0) | (arrays['slope_deg'] > 90)).sum()),
               'negative_roughness': int((arrays['local_elevation_std_m'] < 0).sum()),
               'derivative_without_elevation': int((np.isfinite(arrays['slope_deg']) & ~np.isfinite(z)).sum())}
        if stage.startswith('earth'):
            valid, natural = arrays['valid_input_mask'], arrays['natural_screen_mask']
            row['natural_screen_area_km2'] = float((natural == 1).sum() * cell_area / 1e6)
            row['natural_and_joint_area_km2'] = float(((natural == 1) & joint).sum() * cell_area / 1e6)
            row['screen_without_complete_features'] = int(((natural == 1) & ~joint).sum())
            row['screen_outside_valid'] = int(((natural == 1) & (valid != 1)).sum())
            row['screen_exclusion_violations'] = int(((natural == 1) & ((arrays['built_fraction'] > 0) |
                (arrays['crop_fraction'] > 0) | (arrays['water_fraction'] > 0) |
                (arrays['bare_sparse_fraction'] < .8) | (arrays['source_coverage'] < .99))).sum())
            fractions = ['built_fraction', 'crop_fraction', 'water_fraction', 'bare_sparse_fraction', 'source_coverage']
            row['fraction_range_violations'] = sum(int(((arrays[k] < -1e-6) | (arrays[k] > 1+1e-6)).sum()) for k in fractions)
            row['partial_class_sum_above_one'] = int((sum(arrays[k] for k in fractions[:-1]) > 1.00001).sum())
        else:
            support = arrays['image_support_fraction']
            row['elevation_below_support_threshold'] = int((np.isfinite(z) & (~np.isfinite(support) | (support < .99))).sum())
            row['fraction_range_violations'] = int(((support < -1e-6) | (support > 1+1e-6)).sum())
        terrain_rows.append(row)
        print('Terrain', region, row, flush=True)
        del arrays
    if stage.startswith('lunar'):
        for p in sorted(p for p in base.glob('moon/*/native/*/*') if p.suffix.lower() == '.tif'):
            a = load_raster(p)
            with rasterio.open(p) as src:
                native_rows.append({'region': p.parents[2].name, 'category': p.parent.name,
                    'file': p.name, 'units': src.units[0], 'crs_missing': src.crs is None, **stats(a)})
results['terrain'] = terrain_rows
pd.DataFrame(raster_rows).to_csv(OUT / 'raster_statistics.csv', index=False)
pd.DataFrame(native_rows).to_csv(OUT / 'native_layer_statistics.csv', index=False)
results['previews'] = previews

In [ ]:
# Tile seams: compare native mosaic jumps at integer-degree joins with nearby parallel jumps.
seams = []
base = ROOT / 'data/processed/earth_terrain_landcover' / RUNS['earth_terrain_landcover']
for p in base.glob('earth/*/native_mosaic.tif'):
    with rasterio.open(p) as src:
        for axis, low, high in [('longitude', src.bounds.left, src.bounds.right), ('latitude', src.bounds.bottom, src.bounds.top)]:
            for degree in range(math.ceil(low), math.floor(high) + 1):
                row, col = src.index(degree if axis=='longitude' else (src.bounds.left+src.bounds.right)/2,
                                     degree if axis=='latitude' else (src.bounds.bottom+src.bounds.top)/2)
                index = col if axis=='longitude' else row
                size = src.width if axis=='longitude' else src.height
                if not 25 < index < size-25: continue
                window = Window(index-21, 0, 42, src.height) if axis=='longitude' else Window(0, index-21, src.width, 42)
                a=src.read(1,window=window,masked=True).astype('float32').filled(np.nan)
                changes=np.abs(np.diff(a,axis=1 if axis=='longitude' else 0))
                line=changes[:,20] if axis=='longitude' else changes[20,:]
                nearby=np.delete(changes,20,axis=1 if axis=='longitude' else 0)
                seams.append({'region':p.parent.name,'axis':axis,'degree':degree,
                              'seam_p95_m':float(np.nanquantile(line,.95)),
                              'nearby_p95_m':float(np.nanquantile(nearby,.95)),
                              'seam_valid_fraction':float(np.isfinite(line).mean())})
results['seams']=seams

In [ ]:
# Climate: completeness, summary reconciliation, physical ordering, and cell-level missingness.
base=ROOT/'data/processed/earth_climate'/RUNS['earth_climate']
climate=pd.read_csv(base/'regional_monthly_summary.csv')
climate_checks={'rows':len(climate),'duplicate_area_date_variable':int(climate.duplicated(['area','date','variable']).sum()),'summary_mismatches':0,'invalid_flags':0,'negative_non_temperature_values':0,'tmin_above_tmax_cells':0}
climate_ranges=[]
for area in records['earth_climate']['config']['areas']:
    for year in records['earth_climate']['config']['years']:
        yearly={}
        for variable in records['earth_climate']['config']['variables']:
            frame=pd.read_csv(base/f'earth/{area}/native_monthly/{variable}_{year}.csv.gz')
            yearly[variable]=frame.set_index(['date','longitude','latitude'])['value']
            climate_checks['invalid_flags']+=int((frame['valid']!=np.isfinite(frame['value'])).sum())
            if variable not in ['tmin','tmax']: climate_checks['negative_non_temperature_values']+=int((frame.value<0).sum())
            climate_ranges.append({'area':area,'variable':variable,'year':year,'min':float(frame.value.min()),'max':float(frame.value.max())})
            for date,group in frame.groupby('date'):
                expected=climate[(climate.area==area)&(climate.variable==variable)&(climate.date==date)]
                valid=group[np.isfinite(group.value)]
                weights=np.cos(np.deg2rad(valid.latitude))
                mean=float((valid.value*weights).sum()/weights.sum())
                if len(expected)!=1 or not np.isclose(mean,expected.iloc[0].spatial_weighted_mean,rtol=1e-8,atol=1e-8): climate_checks['summary_mismatches']+=1
        climate_checks['tmin_above_tmax_cells']+=int((yearly['tmin']>yearly['tmax']).sum())
climate_checks['expected_rows']=3*10*12*5
results['climate_checks']=climate_checks
results['climate_coverage']=climate.groupby(['area','variable']).weighted_coverage.agg(['min','max']).reset_index().to_dict('records')
pd.DataFrame(climate_ranges).to_csv(OUT/'climate_ranges.csv',index=False)

In [ ]:
# OSM geometry and identity validation of exported features.
osm=[]
base=ROOT/'data/processed/osm_support'/RUNS['osm_support']
for p in base.glob('earth/*/*.geojson'):
    features_geo=read_json(p)['features']; quality=read_json(p.parent/'quality.json')
    seen=set(); duplicate=invalid=empty=outside=0
    boundary=box(*quality['bbox_lon_lat'])
    for feature in features_geo:
        props=feature['properties']
        key=(props.get('osm_type'),props.get('osm_id'))
        duplicate+=key in seen; seen.add(key)
        geom=shape(feature['geometry'])
        invalid+=not geom.is_valid; empty+=geom.is_empty
        outside+=not boundary.buffer(1e-8).covers(geom)
    osm.append({'region':p.parent.name,'features':len(features_geo),'quality':quality,
                'duplicate_ids':duplicate,'invalid_geometry':invalid,'empty_geometry':empty,'outside_bbox':outside})
results['osm']=osm

In [ ]:
# Diviner: every exported table, unique cells, bounds, valid flags, summary counts and extrema.
base=ROOT/'data/processed/diviner_temperature'/RUNS['diviner_temperature']
summary=pd.read_csv(base/'regional_bin_summary.csv')
checks={'rows':len(summary),'duplicates':int(summary.duplicated(['region','source_product']).sum()),
        'summary_mismatches':0,'duplicate_cells':0,'outside_bbox':0,'invalid_flags':0,'nonpositive_valid_K':0}
for entry in summary.itertuples():
    frame=pd.read_csv(base/f'moon/{entry.region}/native_240m/{entry.source_product}.csv.gz', dtype={'tbol': float, 'valid_temperature': bool})
    checks['duplicate_cells']+=int(frame.duplicated(['clon','clat']).sum())
    west,south,east,north=records['diviner_temperature']['config']['regions'][entry.region]
    checks['outside_bbox']+=int((~(frame.x_m.between(west,east,inclusive='left') & frame.y_m.between(south,north,inclusive='left'))).sum())
    finite=np.isfinite(frame.tbol)
    checks['invalid_flags']+=int((finite!=frame.valid_temperature).sum())
    checks['nonpositive_valid_K']+=int((finite & (frame.tbol<=0)).sum())
    mismatch=len(frame)!=entry.selected_rows or int(finite.sum())!=entry.valid_temperature_rows
    if finite.any():
        mismatch|=not np.allclose([frame.tbol.mean(),frame.tbol.min(),frame.tbol.max()],
            [entry.spatial_mean_of_bin_average_K,entry.spatial_min_of_bin_average_K,entry.spatial_max_of_bin_average_K],rtol=1e-8,atol=1e-8)
    checks['summary_mismatches']+=int(mismatch)
results['diviner_checks']=checks
summary['empty']=summary.valid_temperature_rows==0
results['diviner_coverage']=summary.groupby('region').agg(bins=('source_product','count'),empty_bins=('empty','sum'),min_K=('spatial_min_of_bin_average_K','min'),max_K=('spatial_max_of_bin_average_K','max'),total_rows=('selected_rows','sum')).reset_index().to_dict('records')
summary.groupby(['seasonal_ls_bin','subsolar_longitude_bin']).agg(empty_regions=('empty','sum')).reset_index().to_csv(OUT/'diviner_empty_bins.csv',index=False)
pd.DataFrame(terrain_rows).to_csv(OUT/'terrain_readiness.csv',index=False)
def json_ready(value):
    if isinstance(value, dict):
        return {k: json_ready(v) for k, v in value.items()}
    if isinstance(value, list):
        return [json_ready(v) for v in value]
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


(OUT/'audit.json').write_text(json.dumps(json_ready(results),indent=2,allow_nan=False),encoding='utf-8')
print('Audit saved:',OUT,flush=True)

In [ ]:
# Additional checks independently enforce complete keys and full feature neighborhoods.
from scipy.ndimage import convolve

additional = {'feature_support': [], 'climate': {}, 'diviner': {}}
for stage in ['earth_terrain_landcover', 'lunar_terrain_environment']:
    base = ROOT/'data/processed'/stage/RUNS[stage]
    for grid in sorted(base.glob('*/*/grid_30m')):
        elevation_valid = np.isfinite(load_raster(grid/'elevation_m.tif'))
        row = {'region': grid.parent.name}
        for layer, size in [('slope_deg',3), ('laplacian_per_m',3), ('local_elevation_std_m',11)]:
            # Summed occupancy is independent of the production minimum-filter implementation.
            complete = convolve(elevation_valid.astype('int16'), np.ones((size,size),dtype='int16'),mode='constant',cval=0)==size*size
            row[layer+'_unsupported_cells'] = int((np.isfinite(load_raster(grid/(layer+'.tif'))) & ~complete).sum())
        if stage.startswith('earth'):
            natural = load_raster(grid/'natural_screen_mask.tif')
            valid = load_raster(grid/'valid_input_mask.tif')
            row['nonbinary_masks'] = int((np.isfinite(natural)&~np.isin(natural,[0,1])).sum()+(np.isfinite(valid)&~np.isin(valid,[0,1])).sum())
            finite = np.ones(elevation_valid.shape,dtype=bool)
            for layer in ['built_fraction','crop_fraction','water_fraction','bare_sparse_fraction','source_coverage']:
                finite &= np.isfinite(load_raster(grid/(layer+'.tif')))
            row['screen_with_missing_landcover'] = int(((natural==1)&~finite).sum())
        additional['feature_support'].append(row)

base=ROOT/'data/processed/earth_climate'/RUNS['earth_climate']
summary_c=pd.read_csv(base/'regional_monthly_summary.csv')
config=records['earth_climate']['config']
expected={(a,f'{y}-{m:02d}-01',v) for a in config['areas'] for y in config['years'] for m in range(1,13) for v in config['variables']}
actual=set(summary_c[['area','date','variable']].itertuples(index=False,name=None))
additional['climate']={'missing_summary_keys':sorted(expected-actual),'extra_summary_keys':sorted(actual-expected),
                       'duplicate_cells':0,'missing_table_months':0,'summary_count_or_range_mismatches':0,'coverage_mismatches':0}
for area in config['areas']:
    for year in config['years']:
        for variable in config['variables']:
            f=pd.read_csv(base/f'earth/{area}/native_monthly/{variable}_{year}.csv.gz')
            additional['climate']['duplicate_cells']+=int(f.duplicated(['date','longitude','latitude']).sum())
            additional['climate']['missing_table_months']+=len({f'{year}-{m:02d}-01' for m in range(1,13)}-set(f.date))
            for date,g in f.groupby('date'):
                s=summary_c[(summary_c.area==area)&(summary_c.date==date)&(summary_c.variable==variable)]
                if len(s)!=1: continue
                s=s.iloc[0]; v=g[np.isfinite(g.value)]
                mismatch=len(g)!=s.selected_cells or len(v)!=s.valid_cells
                mismatch|=not np.allclose([v.value.min(),v.value.max()],[s.spatial_min_of_monthly_field,s.spatial_max_of_monthly_field],equal_nan=True)
                additional['climate']['summary_count_or_range_mismatches']+=int(mismatch)
                coverage=np.cos(np.deg2rad(v.latitude)).sum()/np.cos(np.deg2rad(g.latitude)).sum()
                additional['climate']['coverage_mismatches']+=int(not np.isclose(coverage,s.weighted_coverage,atol=1e-8,rtol=1e-8))

record=records['diviner_temperature']
base=ROOT/'data/processed/diviner_temperature'/RUNS['diviner_temperature']
summary_d=pd.read_csv(base/'regional_bin_summary.csv')
products={Path(p).stem for p in record['inputs'] if p.lower().endswith('.xml')}
expected={(region,product) for region in record['config']['regions'] for product in products}
actual=set(summary_d[['region','source_product']].itertuples(index=False,name=None))
additional['diviner']={'input_products':len(products),'missing_summary_keys':sorted(expected-actual),'extra_summary_keys':sorted(actual-expected),
                       'missing_count_mismatches':int((summary_d.missing_temperature_rows!=(summary_d.selected_rows-summary_d.valid_temperature_rows)).sum()),
                       'status_mismatches':int((summary_d.status!=np.where(summary_d.valid_temperature_rows>0,'has_data','no_valid_cells')).sum()),
                       'empty_bins_with_finite_statistics':int(summary_d.loc[summary_d.valid_temperature_rows==0,['spatial_mean_of_bin_average_K','spatial_min_of_bin_average_K','spatial_max_of_bin_average_K']].notna().any(axis=1).sum())}
results['additional_checks']=additional
(OUT/'audit.json').write_text(json.dumps(json_ready(results),indent=2,allow_nan=False),encoding='utf-8')
print('Additional checks:',json.dumps(additional,indent=2),flush=True)


In [ ]:
import importlib.metadata
import platform

runtime = {"python": platform.python_version(),
           "packages": {name: importlib.metadata.version(name) for name in
                        ["numpy", "pandas", "rasterio", "scipy", "shapely", "nbformat"]},
           "notebook_sha256": digest(ROOT / "notebooks/preprocessing/07_validate_processed_outputs.ipynb"),
           "run_provenance_sha256": {stage: digest(ROOT / "data/processed" / stage / run_id / "provenance.json")
                                     for stage, run_id in RUNS.items()},
           "method": "Read-only output audit; no independent source-accuracy or mission-suitability validation."}
(OUT / "audit_provenance.json").write_text(json.dumps(runtime, indent=2), encoding="utf-8")
